# FailureSensorIQ GEPA Optimization Experiments

- In this experiment we utilize open source models like `gpt-oss-120b` to optimize Chain of Thought (CoT) `dspy.ChainOfThought` for solving the `FailureSensorIQ` Datasets MCQA results.   
- In this notebook we will run different combinations hyper parameter tuning

#### Current Approach 
- Using grid search to find an optimal solution
- In the future we can use `optuna` when using model and combinations

In [ ]:
# Library Imports
import os
from datasets import load_dataset
import dspy
from dspy import GEPA

import random
import re
from typing import Optional

from itertools import product

import logging

import json
from pathlib import Path
from datetime import datetime

In [ ]:
# Global Variables & Flags
seed = 42

# ML Flow Configuration
MLFlow_Active = True

if MLFlow_Active:
    # Increase connection pool settings before importing mlflow
    os.environ["MLFLOW_HTTP_POOL_CONNECTIONS"] = "50"  # number of pools
    os.environ["MLFLOW_HTTP_POOL_MAXSIZE"] = "50"  # maximum size of each pool

 # Load FailureSensorIQ Dataset
ds = load_dataset("ibm-research/FailureSensorIQ", "single_true_multi_choice_qa")



In [ ]:
# Watsonx AI Credentials
from ibm_watsonx_ai import APIClient,Credentials
api_key = os.getenv("WATSONX_API_KEY")
url = "https://us-south.ml.cloud.ibm.com"
project_id = os.getenv("WATSONX_PROJECT_ID")

#Testing Credentials 
credentials = Credentials(api_key=api_key, url=url)

client = APIClient(credentials)

client.set.default_project(project_id)

In [ ]:
available_models = client.foundation_models.ChatModels.show()

In [ ]:
# MLFlow Integration with DSPy 

if MLFlow_Active:
    # Setup MLFlow
    import mlflow

    # To Run the Server 
    # mlflow server --backend-store-uri sqlite:///mydb.sqlite

    # Enable autologging with all features
    mlflow.dspy.autolog(
        log_compiles=True,    # Track optimization process
        log_evals=True,       # Track evaluation results
        log_traces_from_compile=True  # Track program traces during optimization
    )

    # Configure MLflow tracking
    mlflow.set_tracking_uri("http://localhost:5000")  # Use local MLflow server
    

In [ ]:
# Helper Functions

def _parse_answer(option_ids: list[str], correct: list[bool]) -> str:
    """
    Convert the dataset's boolean correct list into a letter string.

    Example:
        option_ids = ["A", "B", "C", "D", "E"]
        correct    = [False, False, False, True, False]
        → "D"

        option_ids = ["A", "B", "C"]
        correct    = [True, False, True]
        → "A,C"
    """
    letters = [lid.upper() for lid, flag in zip(option_ids, correct) if flag]
    return ",".join(sorted(letters))


def _build_options_str(options: list[str], option_ids: list[str]) -> str:
    """
    Render options as a lettered multi-line string.

    Example:
        options    = ["partial discharge", "resistance", "current"]
        option_ids = ["A", "B", "C"]
        →  "  A) partial discharge\n  B) resistance\n  C) current"
    """
    return "\n".join(
        f"  {lid}) {text}"
        for lid, text in zip(option_ids, options)
    )


def _collect_and_shuffle(assets: list[str], data_dict: dict[str, list[dspy.Example]]) -> list[dspy.Example]:

    rng = random.Random(seed)
    combined = [ex for a in assets for ex in data_dict.get(a, [])]
    rng.shuffle(combined)
    return combined

def _make_exp_name(params: dict) -> str:
    
    params_abbrev = {
        "model_name"            : lambda v: v.split("/")[-1],          # "gpt-oss-120b"
        "reflection_model_name" : lambda v: v.split("/")[-1],          # "gpt-oss-120b"
        "n_train_assets"        : lambda v: f"ntr{v}",             # "ntr2"
        "test_set_multiplier"  : lambda v: f"tsm{v}",       # "tsm1"
        "temperature"           : lambda v: f"t{v}",                  # "t0.2"
        "reasoning_effort"      : lambda v: v[:3],               # "low" → "low", "medium" → "med", "high" → "hig"
        "gepa_budget"           : lambda v: f"gb{v[:3]}"               # "light" → "gblig", "medium" → "gbmed", "heavy" → "gbhea"   
    }

    exclude_params = {"max_tokens"}

    parts = []
    for k, v in params.items():
        if k in exclude_params:
            continue
        formatter = params_abbrev.get(k, lambda v: f"{k[:3]}{v}")
        parts.append(formatter(v))
    return "_".join(parts)


# Convert any list-like outputs into one-line text to avoid JSON serialization errors
def _list_to_text(v):
    if isinstance(v, list):
        return "[" + ", ".join(str(x) for x in v) + "]"
    return "[" + str(v) + "]"


In [ ]:
# Core Functions & Classes
def init_dataset_splits(ds, n_train_assets,test_set_multiplier,train_val_assets_eq = False):
    
    fsiq_dict: dict[str, list[dspy.Example]] = {}

    # Parsing Rows into dspy.Examples
    for row in ds["train"]:

        q_type = row.get("question_type", "")

        options_str = _build_options_str(row["options"], row["option_ids"])
        answer      = _parse_answer(row["option_ids"], row["correct"])
        asset       = row.get("asset_name", "unknown")

        ex = dspy.Example(
            question      = row["question"],
            options       = options_str,
            answer        = answer,
            asset         = asset,
            query_type    = row.get("relevancy", "unknown"),
            question_type = q_type,
        ).with_inputs("question", "options")
    
        fsiq_dict.setdefault(asset, []).append(ex)

    train_assets     = [asset for i, (asset, examples) in enumerate(fsiq_dict.items()) if i < n_train_assets]
    remaining_assets = [asset for i, (asset, examples) in enumerate(fsiq_dict.items()) if i >= n_train_assets]

    if train_val_assets_eq == True:
        val_assets = train_assets
        test_assets = remaining_assets
    else:
        val_assets  = [a for i, a in enumerate(remaining_assets) if i % 2 == 0]
        test_assets = [a for i, a in enumerate(remaining_assets) if i % 2 == 1]


    print("Data Split Configuration:")
    print("\ttrain_assets :" , train_assets)
    print("\tval_assets   :" , val_assets)
    print("\ttest_assets  :" , test_assets)

    train_set = _collect_and_shuffle(train_assets, fsiq_dict)
    val_set   = _collect_and_shuffle(val_assets, fsiq_dict)
    test_set  = _collect_and_shuffle(test_assets, fsiq_dict)*test_set_multiplier # Default Value =1  

    print(f"\ttrain_set: {len(train_set)} samples")
    print(f"\tval_set: {len(val_set)} samples")
    print(f"\ttest_set: {len(test_set)} samples")

    return train_set, val_set, test_set, train_assets, val_assets, test_assets

class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    question  : str = dspy.InputField(desc="The MCQA question about failure modes for industrial assets.")
    options   : str = dspy.InputField(desc="Lettered answer choices, one per line.")
    reasoning : str = dspy.OutputField(desc="Step-by-step reasoning referencing FMEA knowledge.")
    answer    : str = dspy.OutputField(desc="Correct letter(s), e.g. 'D' or 'A,C'.")


# Evaluation Metric
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    """
    Metric that checks if prediction is one of the valid options (A-E or combinations like A,C)
    and matches the correct answer.
    """
    correct_answer = example['answer']
    
    try:
        llm_answer = str(prediction.answer).strip().upper()
        
        # Validate that answer is one of the valid options (single letter or comma-separated letters A-E)
        # Valid formats: "A", "B", "A,C", "B,D", etc.
        answer_pattern = r'^[A-E](,[A-E])*$'
        
        if not re.match(answer_pattern, llm_answer):
            return 0
        
        # Ensure answers are sorted (e.g., "A,C" not "C,A") to match expected format
        parts = llm_answer.split(',')
        llm_answer_normalized = ','.join(sorted(set(parts)))
        correct_answer_normalized = ','.join(sorted(set(correct_answer.split(','))))
        
    except (ValueError, AttributeError, TypeError):
        return 0
    
    return int(correct_answer_normalized == llm_answer_normalized)

def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    """
    Metric that checks if prediction is one of the valid options (A-E or combinations like A,C)
    and matches the correct answer.
    """
    correct_answer = example['answer']
    
    try:
        llm_answer = str(prediction.answer).strip().upper()
        
        # Validate that answer is one of the valid options (single letter or comma-separated letters A-E)
        # Valid formats: "A", "B", "A,C", "B,D", etc.
        answer_pattern = r'^[A-E](,[A-E])*$'
        
        if not re.match(answer_pattern, llm_answer):
            return 0
        
        # Ensure answers are sorted (e.g., "A,C" not "C,A") to match expected format
        parts = llm_answer.split(',')
        llm_answer_normalized = ','.join(sorted(set(parts)))
        correct_answer_normalized = ','.join(sorted(set(correct_answer.split(','))))
        
    except (ValueError, AttributeError, TypeError):
        feedback_text = f"The final answer must be one of the options (A-E or combinations like A,C). You responded with '{prediction.answer}', which couldn't be parsed as a valid option. Please ensure your answer is a valid option without any additional text or formatting."
        feedback_text += f" The correct answer is '{correct_answer}'."      

        #print(feedback_text)

        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)

    feedback_text = ""
    if score == 1:
        feedback_text = f"Your answer is correct. The correct answer is '{correct_answer}'."
    else:
        feedback_text = f"Your answer is incorrect. The correct answer is '{correct_answer}'."

    return int(correct_answer_normalized == llm_answer_normalized)

In [ ]:
# Core Experiment Function

from matplotlib import text


def run_experiment(exp_params: dict, ds,results_path): 

    # Unpack experiment parameters
    model_name = exp_params["model_name"]
    reflection_model_name = exp_params["reflection_model_name"]
    max_tokens = exp_params["max_tokens"]
    n_train_assets = exp_params["n_train_assets"]
    test_set_multiplier = exp_params["test_set_multiplier"]
    temperature = exp_params["temperature"]
    reasoning_effort = exp_params["reasoning_effort"]
    geba_budget = exp_params["gepa_budget"]
    
    
    exp_name = _make_exp_name(exp_params)

    print(f"Running Experiment: {exp_name}")
    print(f"Experiment Parameters: \n\t{exp_params}")
    
    ''' 
    exp_name_base = "GEPAxFSIQ_Base - " + exp_name
    exp_name_gepa = "GEPAxFSIQ_Optimized - " + exp_name
    '''
    mlflow_exp_name = "GEPAxFSIQ_" + exp_name
 
    # Setup DSPy
    lm = dspy.LM(model=model_name, max_tokens=max_tokens, api_key=api_key,
                temperature = temperature
                ,reasoning_effort = reasoning_effort
                ,project_id = project_id
                ,api_base=url
                )
    
    dspy.configure(lm=lm, trace=[], experimental=True)
    
    # Create Data Splits
    train_set, val_set, test_set, train_assets, val_assets, test_assets = init_dataset_splits(ds,n_train_assets,test_set_multiplier,train_val_assets_eq = True)

    # Evaluating the Unoptimized Baseline Chain of Thought (CoT)
    program = dspy.ChainOfThought(GenerateResponse)
    
    if MLFlow_Active:
        mlflow.set_experiment(mlflow_exp_name)  # Set experiment name

    print("Evaluating Base Metrics:")

    
    evaluate = dspy.Evaluate(
    devset=test_set, #change to test_set for final evaluation
    metric=metric,
    num_threads=16,
    display_table=False,
    display_progress=True
    )

    result_base = evaluate(program)

    '''
    if MLFlow_Active:
        mlflow.set_experiment(exp_name_gepa)  # Set experiment name
    '''

    optimizer = GEPA(
        metric=metric_with_feedback,
        auto=geba_budget, # options ['light', 'medium', 'heavy']
        num_threads=16,
        track_stats=True,
        reflection_minibatch_size=3,
        reflection_lm=dspy.LM(model=reflection_model_name
                                ,max_tokens=max_tokens
                                ,api_key=api_key
                                ,temperature = temperature
                                ,reasoning_effort = reasoning_effort
                                ,project_id = project_id   
                                ,api_base=url)
    )

    optimized_program = optimizer.compile(
        program,
        trainset=train_set,
        valset=val_set
    )

    #Evaluating the Optimized Program with GEPA
    print("Evaluating GEPA Optimized Metrics:")
    result_optimized = evaluate(optimized_program,display_table=False)

    # Saving Prompt
    gepa_prompt = optimized_program.predict.signature.instructions
    prompt_filename = Path("./data/"+datetime.now().strftime("%Y-%m-%d_%H%M%S") + "/" + mlflow_exp_name + "_prompt.txt")
    prompt_filename.parent.mkdir(parents=True, exist_ok=True)
    
    with open(prompt_filename, "w", encoding="utf-8") as prompt_file:
        prompt_file.write(gepa_prompt)    

    train_assets_text = _list_to_text([ex for ex in train_assets])
    val_assets_text = _list_to_text([ex for ex in val_assets])
    test_assets_text = _list_to_text([ex for ex in test_assets])

    # Record results in a JSONL file for later analysis
    record = {"id"              : exp_name,
              "datetime"        : datetime.now().isoformat(),
              "base_score"      : result_base.score,
              "gepa_score"      : result_optimized.score,
              "score_improvement_percentage" : round((result_optimized.score - result_base.score) / result_base.score * 100, 2) if result_base.score != 0 else 0,
              "status"          : "Completed",
              "parameters"      : {
                  **exp_params,
                  "train_set_size" : len(train_set),
                  "val_set_size"   : len(val_set),
                  "test_set_size"  : len(test_set),
                  "train_set"      : train_assets_text,
                  "val_set"        : val_assets_text,
                  "test_set"       : test_assets_text,
              },
              }

    # Append one line per run — safe even if job crashes mid-way
    results_path.parent.mkdir(parents=True, exist_ok=True)
    with results_path.open("a") as f:
        try:
            f.write(json.dumps(record) + "\n")
        except TypeError as e:
            # Fallback to a minimal record if non-serializable objects are still present
            fallback_record = {
                "id": exp_name,
                "datetime": datetime.now().isoformat(),
                "base_score": result_base.score,
                "gepa_score": result_optimized.score,
                "status": "Error",
                "error": str(e),
                "parameters": {
                    **exp_params,
                    "train_set_size": len(train_set),
                    "val_set_size": len(val_set),
                    "test_set_size": len(test_set),
                },
            }
            f.write(json.dumps(fallback_record) + "\n")
    

In [ ]:
# Experiment Configurations

exp_param_grid = {
    "model_name":  ["watsonx/openai/gpt-oss-120b"], #["watsonx/ibm/granite-3-3-8b-instruct"],
    "reflection_model_name": ["watsonx/openai/gpt-oss-120b"],
    "max_tokens": [32000],
    "n_train_assets": [1],
    "test_set_multiplier": [1],
    "temperature": [1],#[0.7,1.0,0.2],
    "reasoning_effort": ["low","medium","high"], 
    "gepa_budget": ["medium"] # options ['light', 'medium', 'heavy']
}

# Creating the experiment combinations
exp_combo_list = [
    dict(zip(exp_param_grid.keys(), vals))
    for vals in product(*exp_param_grid.values())
]

exp_ct = len(exp_combo_list)

# Set Log Level 
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
# Main

results_path = Path("./data/GEPAxFailureSensorIQ_Results_v2.jsonl")

for params in exp_combo_list:
    run_experiment(params, ds, results_path)